# Tokenization & Text Preprocessing

Tokenization is the foundational first step in turning raw text into a format that machine learning models — especially LLMs — can understand. Before anything can be embedded, classified, or generated, it has to be *tokenized*.

This notebook covers:
- What tokens are and why they matter
- How different tokenizers (WordPiece, BPE, SentencePiece) work
- How tokenization affects model input length, truncation, and padding
- Real examples using Hugging Face tokenizers

Plans:
-	How tokenization is implemented
-	How raw text becomes token sequences
-	Why subwords exist
-	BPE merges, vocab lookup etc.
- BPE/WordPiece, vocab files, token IDs, types of tokenization

## What are Tokenizers?
- Extremely important component of the NLP pipeline.
- Translate raw text into data that can be understood and processed by models.
- Models can only process numbers, so tokenizers exactly convert our text inputs to numerical data in actuality. 
- The goal is to find the most meaningful representation (the one that makes the most sense to the model) and the smallest representation.

### Tokenization Algorithms
#### Word-based
- Easy to setup and has few rules.
- Gives decent results.
- All it does is split raw text into words. There are multiple ways to go about this.
  - You can split based on space characters. Example right below here:

In [1]:
tokenized_text = "Jim Henson was a puppeteer".split()
print(tokenized_text)

['Jim', 'Henson', 'was', 'a', 'puppeteer']



  
  - There are also variations of word tokenizers that have extra rules for punctuation. With this kind of tokenizer, we can end up with some pretty large “vocabularies,” where a vocabulary is defined by the total number of independent tokens that we have in our corpus.
  - Visualized below.
  ![Word-based tokenization visualisation](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/word_based_tokenization-dark.svg)
  - What happens is that each word gets assigned an ID, starting with 0 up until the total size of the vocabulary.
  - If you want to cover an entire language via a word-based tokenzier, you'll need to have an ID for each word in the language. This will generate a lot of tokens.
  - For english, there are over 500,000 words in the language, so to build a map from each word to an input ID we’d need to keep track of that many IDs. 
    - It's also worth noting that words like “dog” are represented differently from words like “dogs”, and the model will initially have no way of knowing that “dog” and “dogs” are similar: it will identify the two words as unrelated. The same applies to other similar words, like “run” and “running”, which the model will not see as being similar initially.
  - We also need a custom token to represent words that don't actually exist in our vocabulary.
  - This is known as the "unknown" token, represented as "[UNK]" or "<unk>".
    - Note: its not a good sign if your tokenizer is producing a lot of unknown tokens, because that means it wasn't able to retrieve a sensible representation of a word and you're losing information you could've gotten from that word.
  - The goal when crafting the vocabulary is to do it in such a way that the tokenizer tokenizes as few words as possible into the unknown token.
  - To reduce the amount of unknown tokens, we can use character-based tokens.

#### Character-based tokenizers
- Character-based tokenizers split the text into characters, rather than words. This has two primary benefits:

- The vocabulary is much smaller.
- There are much fewer out-of-vocabulary/unknown tokens, since every word CAN be built from characters.
- But here too some questions arise concerning spaces and punctuation:
![Character tokens](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/character_based_tokenization-dark.svg)

- This approach isn’t perfect either. As the representation is now based on characters instead of words, you can say that it’s less meaningful: each character doesn’t mean a lot on its own, while words do. 
  - However, its worth nothing that this again differs according to the language; in Chinese, for example, each character carries more information than a character in a Latin language like English.
- Another weakness is that we’ll end up with a very large amount of tokens to be processed by our model; a word would only be a single token with a word-based tokenizer, but it can easily turn into 10 or more tokens when converted into characters via a character-based tokenizer!
- To get the best of both worlds, we can use a third technique that combines the two approaches: subword tokenization.

#### Subword-based tokenizer
- Subword tokenization algorithms heavily utilize this principle: frequently used words should not be split into smaller subwords, but rare words should be decomposed into meaningful subwords.
- For example, “annoyingly” can be considered a rare word. It could be decomposed into “annoying” and “ly”. These are both likely to appear more frequently as standalone subwords than the sole word "annoyingly" is likely to appear, while at the same time the meaning of “annoyingly” is preserved by the composite meaning of “annoying” and “ly”.

Here is an example showing how a subword tokenization algorithm would tokenize the sequence “Let’s do tokenization!“:

![Subword-tokenization](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/bpe_subword-dark.svg)

- (where `</w>` means the end of an actual entire full word from the original sequence that was passed through the tokenizer.)
- These subwords provide a lot of semantic meaning: for example, in the example above “tokenization” was split into “token” and “ization”, two tokens that have a semantic meaning (like how semantic meaning is preserved in word-based tokenizers) while being space-efficient (only two tokens are used to represent a long word instead of many tokens like in character-based). 
- This approach allows us to have relatively good coverage with small vocabularies, and close to no unknown tokens. The best of all words.
- Additional note: This approach is especially useful in agglutinative languages such as Turkish, where you can just keep (almost) arbitrarily forming longer and longer complex words by stringing together subwords.

#### Other techniques:
- Obviously, there are also other techniques:
  - Byte-level BPE, as used in GPT-2
  - WordPiece, as used in BERT
  - SentencePiece or Unigram, as used in several multilingual models
- You now have enough knowledge to now use APIs like the Hugging Face tokenizers library.



## Tokenizers library usage
- 

## Encoding
- The process of translating text into numbers that can be understood by models.
- Performed across a 2 step process:
  - Tokenization
  - Conversion to input IDs
- We have already covered tokenization.
- We will now cover how to convert those tokens into numbers so they can be fed, as a tensor, to the model.
- For this, the tokenizer has the vocabulary that we originally talked about.
- This vocabulary was generated during the training of the tokenizer model.
- The vocabulary is essentially a dictionary that maps tokens to numbers (which represent their input ID).
- So once you've tokenized, you can then essentially get a list of input IDs that reflect your tokens that you got from tokenization. This usually done pretty easily by just utilizing methods provided by the tokenizer.
- Here's an example. First, lets tokenize our input:



In [9]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

input_sequence = "Hey guys, how are you doing today? I am doing swimmingly well!"
tokens = tokenizer.tokenize(input_sequence)

print(tokens)

['Hey', 'guys', ',', 'how', 'are', 'you', 'doing', 'today', '?', 'I', 'am', 'doing', 'swimming', '##ly', 'well', '!']


This output above are our tokens. This tokenizer is a subword tokenizer: it splits the words until it obtains tokens that can be represented by its vocabulary. That’s the case here with "swimmingly", which is split into two tokens: swimming and ##ly.

Now, lets convert them to input IDs that would be usable by a tokenizer model/vocabulary.

In [10]:
input_ids = tokenizer.convert_tokens_to_ids(tokens)
print(input_ids)

[4403, 3713, 117, 1293, 1132, 1128, 1833, 2052, 136, 146, 1821, 1833, 5947, 1193, 1218, 106]


- These outputs, once converted to the appropriate framework tensor (so a tensor for tensorflow vs a tensor for pytorch), can then be used as inputs to a model, as seen earlier.
- It is extremely important to note that these input IDs don't actually carry meaning on their own.
- Now, you gets vector embeddings for these tokens. These embeddings are the ones that ACTUALLY carry meaning on their own, and give tokens some sort of semantic meaning.
  - Vector embeddings usually come in the form of something like `[0.12, -0.44, ..., 0.08]`.
  - The length of 1 vector of embeddings is known as the "hidden dimension".
  - The values across that vector are used together to "give meaning" to the token.
  - Different models (BERT, GPT etc) learn meanings for different tokens/words during pre-training.
  - It is EXTREMELY likely that different models will give different tokens different vector embeddings values, even if they have the same meaning to us.
  - Different models use different hidden dimensions.
- How it works:
  - Each input ID is looked up in the embedding matrix
    - The embedding matrix is a giant lookup table
    - The shape of the matrix is [vocab_size x hidden_size] (e.g. [30000 x 512])
    - Each row of the embedding matrix is the learned vector (vector embedding) for one token.
  - The result of the lookup is the full vector embedding for that token and can be used (by the model) to actually understand what that token means and what its semantic meaning is.
  - So example:
    - ID 7993 -> row 7993 in the matrix -> a vector like [0.12, -0.44, ..., 0.08] (of length 512)
    - ID 170 -> row 170 -> another vector
- Once you have all these vector embeddings, all these vector embeddings get stacked into a matrix [sequence_length x hidden_size]. So if you have a sequence length of 7 tokens, your final embedding matrix for the input sentence will be a [7 x 512] matrix.
- This matrix will then be the actual input the transformer layers of the model will use.
- The matrix is specific to the model you are using.
- At the start of pre-training, the model's embedding matrix just has random meaningless values. Over time, as the model is trained and it seems more and more data/tokens, it keeps learning about the meaning about the meaning of different tokens/text/data, and accordingly keeps adjusting the representations for different tokens (which is the embedding matrix) as it iterates over the corpus it is pre-trained on.
- Here's an example where we use the BERT embedding matrix to look up the learned vector for the tokens we generated previously:

In [11]:
from transformers import BertModel

model = BertModel.from_pretrained("bert-base-uncased")
embedding_matrix = model.embeddings.word_embeddings.weight  # shape: [vocab_size, hidden_dim]
print(embedding_matrix.shape)

torch.Size([30522, 768])


As you can see, the vocabulary size uesd by BERT is 30522 tokens! And the hidden dimension used (length for the embedding vector used to store semantic meaning for each token) is 768!

Now lets perform a lookup of the embedding matrix using the token ID for the token "swimming" that we got from earlier: 5947.

In [12]:
id = 5947
vector = embedding_matrix[id]  # shape: [hidden_dim]
print(vector.shape)
print(vector)

torch.Size([768])
tensor([-1.6336e-03,  1.9387e-02, -4.2262e-02, -1.4006e-02, -4.4008e-02,
        -9.2689e-02, -5.0611e-02,  1.7712e-03,  1.8792e-03, -2.1428e-02,
        -6.9403e-03, -8.1696e-02, -7.8513e-03, -1.1460e-02, -2.4959e-02,
        -1.9781e-02, -5.7871e-02, -7.0826e-02,  5.2755e-02,  9.9685e-03,
        -2.9509e-03, -4.5740e-02, -2.2632e-02, -4.9202e-02, -1.0307e-03,
        -4.0803e-02, -2.2185e-02, -6.2010e-02, -3.2440e-03,  1.5996e-02,
        -7.0029e-02, -8.1802e-03,  7.1709e-03, -3.6447e-02, -5.1779e-02,
        -5.4396e-02, -7.5384e-02, -2.3541e-02, -7.2916e-03, -2.4236e-02,
        -3.0795e-02, -2.6845e-04, -7.3996e-03,  1.2830e-03,  5.4537e-02,
        -3.8885e-03,  5.3802e-03, -6.8084e-02,  2.8300e-02, -5.4006e-02,
        -1.8643e-02, -1.0900e-02,  3.0778e-04,  2.8183e-02, -1.0486e-02,
        -3.4683e-02, -3.8088e-02, -4.5153e-03, -1.8099e-02, -1.2556e-02,
        -4.7107e-02, -9.6444e-02,  3.6949e-02, -4.6472e-03, -4.0660e-03,
         5.0509e-02, -8.8737e-02,

And that right there is the vector embedding used to truly store meaning about the word "swimming" within the bert-base-uncased model.

### A note on terminology
- To make sure everything truly sticks in your head, I will clarity on terminology so that things don't get mixed up.
- The usage of "bert-base-uncased' in this previous example is actually just a **bundle identifier** for:
    - The model weights (the actual neural net, i.e., BERT)
	- The tokenizer config (vocab, casing rules, special tokens, etc.)

### Decoding
- Tokenizer's also often come with decoders, so that you can convert token IDs back into their original tokens.
- In the library we've been using so far in this notebook, we can just use the decode() function that comes with the bert-base-uncased tokenizer object we created earlier.
- Here's an example where we use decoding:

In [13]:
decoded_string = tokenizer.decode(input_ids)
print(decoded_string)

Hey guys, how are you doing today? I am doing swimmingly well!


- Notice how the decode method not only converts the indices back to tokens, but also groups together the tokens that were part of the same words to produce an actually human-readable sentence. 
- This behavior will be extremely useful when we use models that predict new text (either text generated from a prompt, or for sequence-to-sequence problems like translation or summarization) because then the model is able to directly output actually human-readable sentences to the user during usage, instead of just token after token (for example, using our earlier example, instead of just writing "...swimming ##ly...", it would write "...swimmingly..." to the user thanks to decoding).


## What do the vector embedding values ACTUALLY mean?

For those of you, like me, who really want to intuitively understand how everything actually works under the hood, you'd want to know what the vector embedding values actually mean. How can a word be represented by a list of 500 to 1000 numbers?

First off, the values in an embedding vector don’t correspond to simple human concepts like “redness” or “happiness.” Each value doesn’t represent a distinct semantic trait. Instead, the entire vector works *together* to capture meaning *relationally* (as in how this token’s meaning relates to others in the corpus).

### How embeddings update
- During training, the model processes sequences of tokens, and tries to perform some task (like next word prediction).
- The output is compared to the correct answer, and a **loss** is calculated.
- Then **backpropagation** kicks in: gradients are passed backward from the output layer, through attention blocks, and finally to the input, which includes the embedding vectors.
- The greater the loss, the greater the gradients passed backwards, the greater/stronger the weight upgrades made in the embedding vectors and attention matrices.
  - Vice versa is also true, if the loss is small, the gradients and therefore updates made will be much smaller and fine-tuned.

This means:
- The vectors for the tokens in each input sequence are nudged slightly to help improve the model’s prediction next time.
- These nudges are determined by the gradient of the loss function with respect to each embedding vector.
- So over time, embeddings shift and rotate in space to better capture how tokens tend to appear in relation to others.

### An example of embedding updates
Let’s say the model is training on the sentence:
> “The apple was picked from the tree.”

Early on:
- All embeddings are random — so “apple”, “tree”, “picked” all point in meaningless directions.
- The model predicts a wrong word after “picked”. This creates a loss.
- Gradients flow backward and *adjust* the embeddings of “apple”, “picked”, and “tree” slightly to move toward a better internal representation.

Later on:
- The model sees more sentences like:
  - “She ate the apple.”
  - “He picked fruit from the backyard.”
  - “The farmer collected apples.”
- Gradually, the embedding for “apple” gets pulled toward the vectors for “fruit”, “tree”, “picked”, “farmer”, etc.

So the embedding of "apple" ends up in a *region* of space that reflects its use — not because any one value means “fruitness” — but because **statistically**, that’s how "apple" is used.

This also means that semantically similar tokens (like "banana", "mango", "apple") will end up close to each other in this high-dimensional space, not by hardcoding, but by emerging from statistical learning.

### But wait, if all the embeddings are random at the start, how does anything ever converge?
This is a great question.

At the beginning, **everything is wrong**; embeddings are random, attention weights are random, everything. But that’s okay!
- The model still makes predictions.
- Even if it’s wrong, the loss is computed.
- Gradients are computed from that loss and flow backward.
- The update might not be in the *perfect* direction, but **it’s still a direction that reduces loss just a bit.**
- Over millions of steps, with enough data, these small updates add up.

So:
- Early updates may be noisy and rough.
- But they still start pulling similar contexts together.
- The model slowly aligns words that tend to occur together into nearby regions in embedding space.

### Bonus: embeddings aren’t static during attention either
- The **input to attention layers is the embedding vectors**.
- As gradients flow through attention heads, the model also learns to adjust:
  - The embeddings themselves
  - The **attention score weights** (the W_q, W_k, W_v matrices, which project embeddings into different spaces)
- So attention and embedding learning happens **together**.

So from: 
- Random initialization
- Cross-entropy loss
- Backpropagation
- Lots of data

→ Semantic space emerges!

That’s just exactly how a token ends up having an expressive, context-aware, learned vector, built entirely from scratch through gradient-based learning over huge amounts of text.

And that's it for tokenization! 